# 00 — Dataset Preparation

Anonymize the original lead dataset so it's safe to share on GitHub.

**Steps**:
1. Load the original CSV (`insurance_leadgen_data_old.csv`)
2. Remove ~5% of non-verified leads
3. Recreate `lead_id` with random 8-digit IDs
4. Simplify column names
5. Save as `insurance_leadgen_data.csv` (this is the version that gets shared)

**Note**: Run this once. The `_old` file is gitignored — only the clean version goes to GitHub.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
DATASETS_DIR = PROJECT_ROOT / "datasets"

## 1 · Load Original Data

In [2]:
leads_original = pd.read_csv(DATASETS_DIR / "insurance_leadgen_data_old.csv")

print(f"Original: {leads_original.shape[0]:,} rows × {leads_original.shape[1]} cols")
print(f"\nColumns: {list(leads_original.columns)}")
print(f"\nLead Status:")
print(leads_original["Lead Status"].value_counts())
print(f"\nVerification Status:")
print(leads_original["Verification Status"].value_counts())

Original: 7,878 rows × 13 cols

Columns: ['Lead ID', 'Lead Status', 'Premium', 'Age', 'Gender', 'Smoker', 'Current Insurance', 'Device Type', 'Keyword', 'Match Type', 'Patrial Postcode', 'COVER FOR', 'Verification Status']

Lead Status:
Lead Status
Contacted     6280
Quoted         533
Sold           386
Invalid        342
No Contact     337
Name: count, dtype: int64

Verification Status:
Verification Status
verified    1642
Name: count, dtype: int64


## 2 · Anonymize

Three steps to make the data safe for public sharing:
1. **Drop ~5% of non-verified leads** — reduces the dataset slightly and removes the weakest leads
2. **Recreate lead_id** — random 8-digit IDs so original records can't be traced
3. **Simplify column names** — cleaner snake_case names

In [3]:
np.random.seed(42)

# Step 1 — Remove ~5% of non-verified leads
not_verified = leads_original[leads_original["Verification Status"] != "Verified"]
n_drop = int(len(not_verified) * 0.05)
drop_idx = not_verified.sample(n=n_drop, random_state=42).index

leads = leads_original.drop(drop_idx).reset_index(drop=True)
print(f"Dropped {n_drop} non-verified leads")
print(f"Remaining: {len(leads):,} rows (was {len(leads_original):,})")

Dropped 393 non-verified leads
Remaining: 7,485 rows (was 7,878)


In [4]:
# Step 2 — Recreate lead_id with random 8-digit IDs
leads["Lead ID"] = np.random.choice(
    range(10_000_000, 99_999_999), size=len(leads), replace=False
)

# Step 3 — Simplify column names
leads = leads.rename(columns={
    "Lead ID": "lead_id",
    "Lead Status": "lead_status",
    "Premium": "premium",
    "Age": "age",
    "Gender": "gender",
    "Smoker": "smoker",
    "Current Insurance": "current_insurance",
    "Device Type": "device_type",
    "Keyword": "keyword",
    "Match Type": "match_type",
    "Patrial Postcode": "postcode",
    "COVER FOR": "cover_for",
    "Verification Status": "verification_status",
})

print(f"\nNew columns: {list(leads.columns)}")
print(f"Sample lead_ids: {leads['lead_id'].head().tolist()}")
leads.head()


New columns: ['lead_id', 'lead_status', 'premium', 'age', 'gender', 'smoker', 'current_insurance', 'device_type', 'keyword', 'match_type', 'postcode', 'cover_for', 'verification_status']
Sample lead_ids: [90809057, 35622726, 29156003, 26436791, 80270348]


,lead_id,lead_status,premium,age,gender,smoker,current_insurance,device_type,keyword,match_type,postcode,cover_for,verification_status
0,90809057,Contacted,0.0,31,Female,Yes,no,Desktop,private health insurance belfast,Exact,Bt13,Self,NaN
1,35622726,Contacted,0.0,35,Female,No,no,Smartphone,private medical insurance,Phrase,LU7,Self,NaN
2,29156003,Contacted,0.0,46,Male,No,yes private,Smartphone,private health insurance,Broad,E14,Self,NaN
3,26436791,No Contact,0.0,53,Female,No,no,Desktop,bupa health insurance,Exact,W2,Self + Partner,NaN
4,80270348,Contacted,0.0,35,Male,No,no,Smartphone,private healthcare prices,Exact,BH23,Self,NaN


## 3 · Save

In [5]:
leads.to_csv(DATASETS_DIR / "insurance_leadgen_data.csv", index=False)

print(f"Saved → datasets/insurance_leadgen_data.csv ({len(leads):,} rows)")
print(f"\nThis is the version that goes to GitHub.")
print(f"The _old file is gitignored.")

Saved → datasets/insurance_leadgen_data.csv (7,485 rows)

This is the version that goes to GitHub.
The _old file is gitignored.
